# Module 2 — Code Along: Data Structures, Files & Modules (the bank-accounts story)

In Module 1 we modeled **one** account as a `dict` and learned to loop, branch, compute, and fail clearly over a few of them.

A real bank has **many** accounts — and they must survive a restart. So Module 2 is about **holding many accounts in memory** (the four containers) and **storing & loading** them (JSON, CSV), then organizing the code (modules, venvs) and observing it (logging).

Same canonical account shape as M1 — reuse it identically:

```python
id: int, owner: str, account_type: str, balance: float, is_active: bool, tags: list[str]
```

Run top to bottom. Every cell is self-contained, re-runnable, and writes only to a **temp directory** (never the course `data/` dirs).

## The four containers — what are they?

Python ships four built-in collections. Pick by the *question you ask most*: walk in order → **list**; look up by key → **dict**; a fixed never-changing record → **tuple**; "is it in the set / are these unique?" → **set**.

| container | use it for | gotcha |
|---|---|---|
| `list` | ordered, mutable, dupes OK | O(n) lookup |
| `dict` | **lookup by key** (the JSON model) | keys must be hashable |
| `tuple` | fixed-shape immutable record | can't change it (a feature) |
| `set` | unique membership | unordered |

In [ ]:
# A BANK = a LIST of accounts (ordered, we may have duplicates of a value like balance).
accounts = [
    {"id": 1, "owner": "Ada", "account_type": "savings",  "balance": 1500.0, "is_active": True,  "tags": ["primary", "online"]},
    {"id": 2, "owner": "Lin", "account_type": "checking", "balance": 0.0,    "is_active": False, "tags": []},
    {"id": 3, "owner": "Sam", "account_type": "savings",  "balance": 40.0,   "is_active": True,  "tags": ["online"]},
]
print(len(accounts), "accounts; first owner:", accounts[0]["owner"])  # 3 accounts; first owner: Ada

# Each ACCOUNT is a DICT: named fields, looked up by key (this is exactly the JSON shape).
ada = accounts[0]
print(ada["owner"], "->", ada["balance"])      # Ada -> 1500.0

# A TUPLE is a fixed, immutable record — perfect for a thing that must never change,
# e.g. a (year, month, day) the account was opened. WHY tuple: you can't accidentally edit it.
opened_on = (2021, 6, 1)
print("opened:", opened_on)                    # opened: (2021, 6, 1)
try:
    opened_on[0] = 2099                        # tuples reject mutation
except TypeError as err:
    print("immutable:", err)                   # immutable: 'tuple' object does not support item assignment

# A SET holds UNIQUE values, unordered — great for tags with duplicates collapsed.
raw_tags = ["online", "primary", "online", "vip", "vip"]
unique_tags = set(raw_tags)
print(len(unique_tags), "unique tags;", "vip" in unique_tags)  # 4 unique tags; True


## Dict — go deep

The dict is the workhorse: safe lookup with `.get` (no `KeyError`), build an `{id: account}` **index** for O(1) lookup, **merge** two dicts with `|`, and iterate with `.items()`.

The classic bug: **mutating a dict while iterating it** raises `RuntimeError`. Fix: iterate over a *copy* of the keys.

In [ ]:
accounts = [
    {"id": 1, "owner": "Ada", "balance": 1500.0},
    {"id": 2, "owner": "Lin", "balance": 0.0},
    {"id": 3, "owner": "Sam", "balance": 40.0},
]

# .get returns a default instead of raising KeyError when the key is missing.
ada = accounts[0]
print(ada.get("owner"))                 # Ada
print(ada.get("overdraft", "none set")) # none set  — WHY .get: no crash on absent field

# Build an {id: account} INDEX so lookups are by key, not a linear scan.
by_id = {a["id"]: a for a in accounts}
print(by_id[2]["owner"])                # Lin — O(1) lookup by id

# MERGE two dicts with | (3.9+). Right side wins on key clashes — here we patch Ada's balance.
patch = {1: {"id": 1, "owner": "Ada", "balance": 1600.0}}
merged = by_id | patch
print(merged[1]["balance"])             # 1600.0

# ITERATE keys+values together with .items().
for acct_id, acct in by_id.items():
    print(acct_id, acct["owner"])       # 1 Ada / 2 Lin / 3 Sam

# GOTCHA: deleting while iterating the SAME dict raises. Iterate a COPY of the keys instead.
balances = {a["id"]: a["balance"] for a in accounts}
for acct_id in list(balances):          # list(...) snapshots the keys first
    if balances[acct_id] == 0.0:
        del balances[acct_id]           # safe: we're not iterating `balances` itself
print(balances)                         # {1: 1500.0, 3: 40.0} — the zero-balance account is gone


## Comprehensions — the Python idiom

A comprehension builds a new collection from an existing one in one line — it says *what* you want, not *how* to loop. Three flavors: **list** `[...]`, **dict** `{k: v ...}`, **set** `{...}`.

In [ ]:
accounts = [
    {"id": 1, "owner": "Ada", "balance": 1500.0, "is_active": True,  "tags": ["primary", "online"]},
    {"id": 2, "owner": "Lin", "balance": 0.0,    "is_active": False, "tags": ["online"]},
    {"id": 3, "owner": "Sam", "balance": 40.0,   "is_active": True,  "tags": ["online", "vip"]},
]

# LIST comprehension with a filter: owners of active accounts only.
active_owners = [a["owner"] for a in accounts if a["is_active"]]
print(active_owners)                    # ['Ada', 'Sam']

# DICT comprehension: build {id: balance} — replaces a 3-line for-loop.
balances = {a["id"]: a["balance"] for a in accounts}
print(balances)                         # {1: 1500.0, 2: 0.0, 3: 40.0}

# SET comprehension: every distinct tag across ALL accounts (duplicates auto-collapsed).
all_tags = {tag for a in accounts for tag in a["tags"]}
print(sorted(all_tags))                 # ['online', 'primary', 'vip'] — sorted() for stable output


## JSON — save & load a list of accounts

`json.dump` writes a Python object to a file; `json.load` reads it back. JSON is the natural format for our dict-shaped accounts.

**Note:** JSON has no tuple type — a tuple is written as an array and **round-trips back as a list**. Watch that below.

In [ ]:
import json, tempfile, os

accounts = [
    {"id": 1, "owner": "Ada", "account_type": "savings",  "balance": 1500.0, "is_active": True,  "tags": ("primary", "online")},
    {"id": 2, "owner": "Lin", "account_type": "checking", "balance": 0.0,    "is_active": False, "tags": []},
]

# Use a TEMP dir so re-running this notebook never touches the course data/ dirs.
tmpdir = tempfile.mkdtemp()
path = os.path.join(tmpdir, "accounts.json")

# DUMP: write the list of accounts to disk. indent=2 keeps the file human-readable.
with open(path, "w") as fh:
    json.dump(accounts, fh, indent=2)

# LOAD: read it back into fresh Python objects.
with open(path, "r") as loaded_fh:
    loaded = json.load(loaded_fh)

print(loaded[0]["owner"], loaded[0]["balance"])   # Ada 1500.0 — survived the round-trip

# GOTCHA: we saved Ada's tags as a TUPLE, but JSON has no tuples — it comes back as a LIST.
print(type(accounts[0]["tags"]), "->", type(loaded[0]["tags"]))  # <class 'tuple'> -> <class 'list'>

os.remove(path); os.rmdir(tmpdir)   # cleanup so the run is idempotent


## CSV — rows in, rows out

**❓ Predict first:** after a round-trip through CSV, what *type* is the balance — float or str?

`csv.DictWriter` writes each account dict as a row; `csv.DictReader` reads rows back as dicts. **Always pass `newline=""`** when opening the file — otherwise you get blank lines on Windows.

**The pain:** CSV is pure text — *everything* reads back as a **string** (`"1500.0"`, not `1500.0`). You must coerce types by hand. (M3 + Pydantic on Day 2 solve this automatically.)

In [ ]:
import csv, tempfile, os

# CSV is flat: drop the list-valued `tags` field for this demo; keep scalar fields.
accounts = [
    {"id": 1, "owner": "Ada", "account_type": "savings",  "balance": 1500.0, "is_active": True},
    {"id": 2, "owner": "Lin", "account_type": "checking", "balance": 0.0,    "is_active": False},
]
fields = ["id", "owner", "account_type", "balance", "is_active"]

tmpdir = tempfile.mkdtemp()
path = os.path.join(tmpdir, "accounts.csv")

# WRITE: newline="" is REQUIRED — without it Windows inserts a blank line between rows.
with open(path, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=fields)
    writer.writeheader()
    writer.writerows(accounts)

# READ back as dicts.
with open(path, "r", newline="") as fh:
    rows = list(csv.DictReader(fh))

print(rows[0]["owner"])              # Ada
# THE PAIN: balance came back as the STRING "1500.0", not the float 1500.0.
print(repr(rows[0]["balance"]))      # '1500.0'  — a str, not a float
print(type(rows[0]["balance"]))      # <class 'str'>

# So you must coerce by hand. (Day 2's Pydantic does this for you.)
print(float(rows[0]["balance"]) + 100)  # 1600.0 — works only after float(...)

os.remove(path); os.rmdir(tmpdir)   # cleanup -> idempotent


## Modules & virtual environments — what are they?

A **module** is just a `.py` file; `import` pulls its functions/classes into another file so code stays organized and reusable. A **virtual environment** is a private, per-project Python with its own installed packages — so Project A's dependencies never collide with Project B's.

```bash
python -m venv .venv                 # create it
source .venv/bin/activate            # activate (Windows: .venv\Scripts\activate)
pip install -e ".[dev]"              # install this project's deps into it
```

In [ ]:
import tempfile, os, shutil, importlib.util, sys

# Demo: write a tiny module to a temp dir, then IMPORT it — proving a module is just a .py file.
tmpdir = tempfile.mkdtemp()
mod_path = os.path.join(tmpdir, "bank_utils.py")
with open(mod_path, "w") as fh:
    fh.write(
        "def total_balance(accounts):\n"
        "    # sum balances of active accounts only\n"
        "    return sum(a['balance'] for a in accounts if a['is_active'])\n"
    )

# Load that file as a module named `bank_utils` (what `import bank_utils` does under the hood).
spec = importlib.util.spec_from_file_location("bank_utils", mod_path)
bank_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bank_utils)

accounts = [
    {"owner": "Ada", "balance": 1500.0, "is_active": True},
    {"owner": "Lin", "balance": 200.0,  "is_active": False},
]
# Call the imported function — the helper now lives in its own module, reusable from anywhere.
print(bank_utils.total_balance(accounts))   # 1500.0 (Lin is inactive, excluded)

# A venv is conceptual here, but you can SEE which Python is running:
print("python lives at:", os.path.basename(sys.executable))   # e.g. python3 / python

shutil.rmtree(tmpdir)   # cleanup (rmtree clears the import's __pycache__ too) -> idempotent


## Logging (not `print`) — what is it?

`print` is for scripts; real code uses the `logging` module. A logger has **levels** — `DEBUG < INFO < WARNING < ERROR` — so you can dial verbosity without deleting lines. Loggers are *named* (pass `__name__`) so you can tune each module independently.

In [ ]:
import logging

# Configure once: show INFO and above, with a readable format. force=True so re-running is clean.
logging.basicConfig(level=logging.INFO,
                    format="%(levelname)s %(name)s: %(message)s",
                    force=True)
logger = logging.getLogger("bank")     # a NAMED logger — tune "bank" separately later

def withdraw(acct, amount):
    # Pass values as %s ARGS (not f-strings): lazy formatting + structured logs.
    logger.info("withdraw owner=%s amount=%s", acct["owner"], amount)
    if amount > acct["balance"]:
        logger.warning("declined: insufficient funds for %s", acct["owner"])
        return acct["balance"]
    acct["balance"] -= amount
    return acct["balance"]

ada = {"owner": "Ada", "balance": 1500.0}
print("new balance:", withdraw(ada, 500))    # logs an INFO line, then prints 1000.0
print("new balance:", withdraw(ada, 9999))   # logs INFO + a WARNING (declined), balance unchanged
# DEBUG lines are suppressed at level=INFO — change the level, not your code, to see them.
logger.debug("this DEBUG line is hidden at level=INFO")


## Next: Module 3

We can now hold **many** accounts in the right container, **store and load** them (JSON, CSV), split helpers into **modules**, and **log** what happens.

But an account is still a bare `dict` — nothing stops a negative balance or a typo'd field, and the behavior (`deposit`, `withdraw`) lives in loose functions.

**Next we'll give an account real behavior by wrapping it in a class** — `class BankAccount` with `__init__`, methods, and dunder methods. Same six fields, now with structure.